In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
import os

# --- IMPORT DE PYTORCH ---
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Détection automatique du GPU (Nvidia CUDA, Apple MPS, ou CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

# --- 1. ARCHITECTURE GRU (Conforme au papier de recherche) ---

class StockPredictionGRU(nn.Module):
    def __init__(self, input_size=1, hidden_size=100, num_layers=2, dropout_p=0.2):
        super(StockPredictionGRU, self).__init__()
        
        # Deux couches GRU empilées de 100 unités avec Dropout
        self.gru = nn.GRU(
            input_size=input_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout_p
        )
        
        # Conformément à la formule : W2 * ReLU(W1 * h + b1) + b2
        # La matrice W1 mappe l'état caché vers une dimension intermédiaire (par ex: 50)
        self.fc1 = nn.Linear(hidden_size, 50)
        self.relu = nn.ReLU()
        # La matrice W2 mappe vers la prédiction finale (1 sortie)
        self.fc2 = nn.Linear(50, 1)

    def forward(self, x):
        # x est de dimension (batch_size, seq_length, input_size)
        out, _ = self.gru(x)
        
        # On ne s'intéresse qu'à la sortie du tout dernier pas de temps (t)
        out = out[:, -1, :]
        
        # W1 * h + b1 -> ReLU -> W2 * h + b2
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        
        return out

# --- 2. FONCTION DE CRÉATION DES SÉQUENCES ---

def create_sequences(data, seq_length):
    """
    Transforme une série temporelle 1D en séquences de taille seq_length
    Ex: [1, 2, 3, 4, 5] avec seq_length=3 -> X=[[1,2,3], [2,3,4]], y=[4, 5]
    """
    xs = []
    ys = []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(data[i + seq_length])
    return np.array(xs), np.array(ys)

# --- 3. WALK-FORWARD CROSS VALIDATION AVEC DEEP LEARNING ---

def wfcv_gru(y, seq_length=8, step_size=50, fold_size=200, epochs=30, batch_size=32):
    y_vals = y.values
    n = len(y_vals)
    
    preds = []
    truths = []
    mse_tab = []
    
    # On avance de step_size en step_size
    for start in range(0, n - fold_size - step_size + 1, step_size):
        # Les données pures pour ce fold
        train_data_raw = y_vals[start : start + fold_size]
        
        # Pour prédire le test, il nous faut les "seq_length" jours précédents en historique
        test_start_idx = start + fold_size - seq_length
        test_end_idx = start + fold_size + step_size
        test_data_raw = y_vals[test_start_idx : test_end_idx]
        
        # 1. Standardisation (L'algorithme demande un InverseStandardize à la fin)
        scaler = StandardScaler()
        train_scaled = scaler.fit_transform(train_data_raw.reshape(-1, 1))
        test_scaled = scaler.transform(test_data_raw.reshape(-1, 1))
        
        # 2. Création des séquences glissantes
        X_train, y_train = create_sequences(train_scaled, seq_length)
        X_test, y_test = create_sequences(test_scaled, seq_length)
        
        # Conversion en Tensors PyTorch
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
        
        # Création du DataLoader pour l'entraînement par lots (batches)
        train_data = TensorDataset(X_train_t, y_train_t)
        train_loader = DataLoader(train_data, shuffle=False, batch_size=batch_size)
        
        # 3. Initialisation du Modèle, de l'Optimiseur et de la Fonction de perte
        model = StockPredictionGRU(input_size=1, hidden_size=100, num_layers=2, dropout_p=0.2).to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
        # 4. Entraînement du réseau de neurones
        model.train()
        for epoch in range(epochs):
            for batch_x, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
        # 5. Prédiction sur l'échantillon Test
        model.eval()
        with torch.no_grad():
            test_preds_scaled = model(X_test_t).cpu().numpy()
            
        # 6. InverseStandardize (Conforme à la ligne 5 de l'algorithme 2)
        test_preds = scaler.inverse_transform(test_preds_scaled).flatten()
        test_true = scaler.inverse_transform(y_test).flatten()
        
        preds.extend(test_preds)
        truths.extend(test_true)
        mse_tab.append(mean_squared_error(test_true, test_preds))
        
    return np.array(preds), np.array(truths), np.array(mse_tab)

# --- 4. FONCTION D'ÉVALUATION PAR TICKER ---

def stats_forecasting_gru(name_ticker, df_all, seq_length=8, step_size=50, fold_size=200):
    df_t = df_all[['Date', name_ticker]].copy().rename(columns={name_ticker: 'return'})
    
    df_t = df_t[df_t['return'] > -1].copy()
    df_t['log_return'] = np.log1p(df_t['return']).replace([np.inf, -np.inf], np.nan)
    df_t = df_t.dropna(subset=['log_return'])
    y = df_t['log_return']
    
    # Vérification qu'on a assez de données pour le Deep Learning
    if len(y) < fold_size + step_size + seq_length:
        return None 
    
    # On entraîne le GRU avec 30 epochs (Tu pourras ajuster ce paramètre selon ton temps)
    y_pred, y_true, mse_tab = wfcv_gru(
        y, 
        seq_length=seq_length, 
        step_size=step_size, 
        fold_size=fold_size, 
        epochs=30, 
        batch_size=32
    )

    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() >= 3:
        # Comme on a des prédictions générées par DL, la variance peut parfois être très faible.
        # On capte les erreurs si la matrice OLS n'est pas inversible.
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                reg = sm.OLS(y_true[mask], sm.add_constant(y_pred[mask])).fit()
                ols_r2 = float(reg.rsquared)
                ols_intercept = float(reg.params[0])
                ols_slope = float(reg.params[1])
                p_int = float(reg.pvalues[0]) if len(reg.pvalues) > 0 else np.nan
                p_slope = float(reg.pvalues[1]) if len(reg.pvalues) > 1 else np.nan
        except:
            ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')
    else:
        ols_r2 = ols_intercept = ols_slope = p_int = p_slope = float('nan')

    return {
        'SYMBOL': name_ticker,
        'Model': f'Stacked_GRU_seq{seq_length}',
        'seq_length': seq_length,
        'MSE': float(np.nanmean(mse_tab)) if len(mse_tab) > 0 else np.nan,
        'OLS_R2': ols_r2,
        'OLS_Intercept': ols_intercept,
        'OLS_Slope': ols_slope,
        'OLS_P_Value_Intercept': p_int,
        'P_Value_Slope': p_slope,
        'n_obs': int(len(y)),
    }

# --- 5. EXÉCUTION PRINCIPALE avec TOUS LES TICKERS---
'''
if __name__ == "__main__":
    tqdm.write(f"🚀 PyTorch utilise l'appareil : {device.type.upper()}")
    
    STEP_SIZE = 50
    FOLD_SIZE = 1250 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    all_tickers = df_all.columns[1:].tolist()

    # Le papier indique R^8 en entrée. On va donc utiliser un historique de 8 pas de temps (jours)
    seq_lengths_to_test = [8]
    
    os.makedirs('Resultats', exist_ok=True)

    for seq_len in seq_lengths_to_test:
        nom_modele = f"GRU_Stacked_seq{seq_len}"
        output_file = f"Resultats/resultats_{nom_modele}_all_tickers.csv"
        rows_for_config = []
        
        for name in tqdm(all_tickers, desc=f'Entraînement {nom_modele}'):
            res = stats_forecasting_gru(
                name_ticker=name, 
                df_all=df_all, 
                seq_length=seq_len,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            
            if res is not None:
                rows_for_config.append(res)
            
        df_config = pd.DataFrame(rows_for_config)
        df_config.to_csv(output_file, index=False)
        
        tqdm.write(f"✅ Fichier sauvegardé : {output_file}")
        
    print("\nTous les calculs Deep Learning sont terminés !")
'''
# --- 5. EXÉCUTION PRINCIPALE avec les TESTS TICKERS ---

if __name__ == "__main__":
    tqdm.write(f"🚀 PyTorch utilise l'appareil : {device.type.upper()}")
    
    STEP_SIZE = 50
    FOLD_SIZE = 1250 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    
    test_tickers = [
        "GC1 Comdty", 
        "BLQ2TRUU Index", 
        "GBPJPY Curncy", 
        "M4EUHDVD Index", 
        "HFRXM Index", 
        "GB1M Index", 
        "QW2I Index", 
        "FNER Index", 
        "SPX Index" 
    ]
    tickers_name = "test_tickers"

    seq_lengths_to_test = [8]
    os.makedirs('Resultats', exist_ok=True)

    for seq_len in seq_lengths_to_test:
        nom_modele = f"GRU_Stacked_seq{seq_len}"
        
        # On utilise ta variable tickers_name pour nommer le fichier proprement
        output_file = f"Resultats/resultats_{tickers_name}_{nom_modele}.csv"
        rows_for_config = []
        
        # On remplace all_tickers par test_tickers dans la boucle
        for name in tqdm(test_tickers, desc=f'Entraînement {nom_modele}'):
            res = stats_forecasting_gru(
                name_ticker=name, 
                df_all=df_all, 
                seq_length=seq_len,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            
            if res is not None:
                rows_for_config.append(res)
            
        df_config = pd.DataFrame(rows_for_config)
        df_config.to_csv(output_file, index=False)
        
        tqdm.write(f"✅ Fichier test sauvegardé : {output_file}")
        
    print("\nTous les tests Deep Learning sont terminés !")

🚀 PyTorch utilise l'appareil : CPU


Entraînement GRU_Stacked_seq8: 100%|██████████| 9/9 [2:33:26<00:00, 1022.94s/it]  

✅ Fichier test sauvegardé : Resultats/resultats_test_tickers_GRU_Stacked_seq8.csv

Tous les tests Deep Learning sont terminés !
